# Antarctic slope current from along-slope velocities

The horizontal velocity projected onto the local bathymetric
contour, separating the along-slope flow (the Antarctic Slope
Current) from the cross-slope flow that exchanges water with the
shelf:

$$u_{\rm along} = u\frac{h_y}{|\nabla h|}
- v\frac{h_x}{|\nabla h|}, \qquad
v_{\rm cross} = u\frac{h_x}{|\nabla h|}
+ v\frac{h_y}{|\nabla h|}$$

The original takes the bathymetric gradient with `xgcm` on MOM5's
B-grid. On the regular grid that `regrid` produces, the gradient
is a central difference with spherical metrics and `uo`, `vo` and
`deptho` are already colocated.

`deptho` must go through the *same* target grid as the velocities,
or the projection mixes fields from different grids.

---

Translated from the COSIMA recipe
[`Along-slope-velocities.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Along-slope-velocities.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
import cartopy.crs as ccrs
from esmvalcore.preprocessor import (climate_statistics,
                                     extract_region, regrid)

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Load the data

In [ ]:
common = dict(project='CMIP6', exp='historical',
              ensemble='r1i1p1f1', grid='gn',
              dataset='ACCESS-CM2')

uo = Dataset(short_name='uo', mip='Omon', timerange='2000/2009',
             **common)
vo = uo.copy(short_name='vo')
deptho = Dataset(short_name='deptho', mip='Ofx', **common)

def southern(cube):
    cube = regrid(cube, target_grid='0.5x0.5', scheme='linear')
    return extract_region(cube, start_longitude=0.,
                          end_longitude=360.,
                          start_latitude=-80., end_latitude=-55.)

uo_cube = southern(climate_statistics(uo.load(), operator='mean',
                                      period='full'))
vo_cube = southern(climate_statistics(vo.load(), operator='mean',
                                      period='full'))
deptho_cube = southern(deptho.load())

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def lat_2d(cube):
    """2D latitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("latitude")
    if coord.ndim == 2:
        return coord.points
    lon = cube.coord("longitude").points
    return np.broadcast_to(coord.points[:, None], (coord.shape[0], lon.size))


def guess_bounds(cube, coords=("latitude", "longitude")):
    """Add bounds in place where they are missing."""
    for name in coords:
        if cube.coords(name) and not cube.coord(name).has_bounds():
            try:
                cube.coord(name).guess_bounds()
            except ValueError:  # scalar or single-point coordinate
                pass
    return cube


def depth_coord_name(cube):
    """Name of the vertical coordinate, or None for a 2D field."""
    for name in ("depth", "ocean_sigma_z", "olevel", "lev"):
        if cube.coords(name):
            return name
    for coord in cube.coords(dim_coords=True):
        if coord.units.is_convertible("m") and coord.name() != "latitude":
            return coord.name()
    return None


def surface_field(cube):
    """Shallowest level of a 3D field; a no-op for 2D fields."""
    name = depth_coord_name(cube)
    if name is None:
        return cube
    depth = cube.coord(name)
    index = int(np.argmin(np.abs(depth.points)))
    constraint = iris.Constraint(**{name: depth.points[index]})
    return cube.extract(constraint)


def horizontal_slice(cube):
    """First slice of a cube over its two horizontal dimensions.

    ``cube.slices(["latitude", "longitude"])`` cannot be used here: on a
    curvilinear grid the two coordinates share both dimensions and iris
    rejects them as non-orthogonal.  Slicing by dimension index works
    for regular and curvilinear grids alike.
    """
    horizontal_dims = sorted(
        set(cube.coord_dims("latitude")) | set(cube.coord_dims("longitude")))
    if cube.ndim == len(horizontal_dims):
        return cube
    index = tuple(slice(None) if dim in horizontal_dims else 0
                  for dim in range(cube.ndim))
    return cube[index]


def spherical_polygon_area(lon_vertices, lat_vertices, radius=EARTH_RADIUS):
    """Area in m2 of spherical polygons given their corner coordinates.

    ``lon_vertices`` and ``lat_vertices`` are in degrees with the corners
    along the last axis, which is the ``(..., 4)`` layout CMOR uses for
    ``vertices_longitude`` / ``vertices_latitude``.  Any number of
    corners works as long as they are ordered around the cell.

    The area is the spherical excess, accumulated over a triangle fan
    from the first corner, using the Van Oosterom & Strackee (1983)
    form:

        tan(E/2) = |v1 . (v2 x v3)|
                   / (1 + v1.v2 + v2.v3 + v3.v1)

    with ``v`` the unit vectors of the corners.  This is *exact* for
    great-circle edges and therefore invariant under rotation of the
    sphere, which matters because a tripolar grid is exactly a rotated
    and stretched one: a formula that is only correct for cells aligned
    with latitude circles would degrade precisely where the grid stops
    being aligned.

    The obvious alternative, the spherical shoelace formula
    ``R^2/2 sum (dlon)(sin lat_i + sin lat_{i+1})``, treats each edge as
    linear in ``(lon, sin lat)`` rather than as a great circle.  It is
    exact for a latitude-longitude box and accurate to second order for
    small cells, but it is not rotation invariant and it fails badly for
    cells spanning large longitude ranges, such as those beside a
    displaced pole.

    The two disagree by about 2e-4 relative on a 2-degree cell, and
    neither is "wrong": they assume different cell edges.  A cell whose
    north and south edges follow *parallels* has the shoelace area,
    which is what ``iris.analysis.cartography.area_weights`` returns; a
    cell with great-circle edges has this one.  Both tile the sphere
    exactly.  The discrepancy is far smaller than the difference
    between either and the model's own ``areacello``, which is why
    ``areacello`` is still preferred when it is published.
    """
    lon = np.deg2rad(np.asarray(lon_vertices, dtype=float))
    lat = np.deg2rad(np.asarray(lat_vertices, dtype=float))
    unit = np.stack([np.cos(lat) * np.cos(lon),
                     np.cos(lat) * np.sin(lon),
                     np.sin(lat)], axis=-1)

    first = unit[..., 0, :]
    excess = np.zeros(unit.shape[:-2])
    for corner in range(1, unit.shape[-2] - 1):
        second = unit[..., corner, :]
        third = unit[..., corner + 1, :]
        triple = np.abs(np.einsum("...i,...i->...",
                                  first, np.cross(second, third)))
        denominator = (1.0
                       + np.einsum("...i,...i->...", first, second)
                       + np.einsum("...i,...i->...", second, third)
                       + np.einsum("...i,...i->...", third, first))
        excess += 2.0 * np.arctan2(triple, denominator)
    return excess * radius**2


def area_from_bounds(cube, radius=EARTH_RADIUS):
    """Cell area in m2 from 2D coordinate bounds, or ``None``.

    This is what makes a tripolar grid workable without ``areacello``.
    CMOR requires bounds on ``latitude`` and ``longitude``, and on a
    curvilinear grid those bounds are the four corner vertices of every
    cell, so the area follows exactly from
    :func:`spherical_polygon_area`.

    ESMValCore does not do this: ``try_adding_calculated_cell_area``
    handles regular and rotated-pole grids and raises
    ``CoordinateMultiDimError`` for everything else.  The information is
    in the file regardless.

    Returns ``None`` when the cube has no usable 2D bounds, so callers
    can fall back.
    """
    horizontal = horizontal_slice(cube)
    lat_coord = horizontal.coord("latitude")
    lon_coord = horizontal.coord("longitude")
    if lat_coord.ndim != 2 or lon_coord.ndim != 2:
        return None
    if lat_coord.bounds is None or lon_coord.bounds is None:
        return None
    if lat_coord.bounds.shape[-1] < 3:
        return None
    return np.ma.masked_invalid(
        spherical_polygon_area(lon_coord.bounds, lat_coord.bounds,
                               radius=radius))


def cell_area(cube):
    """Cell area in m2, preferring the model's own ``areacello``.

    Order of preference:

    1. an ``areacello`` cell measure attached to the cube, which is what
       ESMValCore builds from a ``supplementary_variables`` entry.  This
       is the model's own area and is always the best answer: it
       accounts for the actual grid generation, including any partial
       cells at the coast.
    2. a spherical polygon area from the 2D corner vertices, via
       :func:`area_from_bounds`.  CMOR mandates bounds on latitude and
       longitude, so this works on a tripolar grid with no
       ``areacello`` at all.  It assumes great-circle cell edges on a
       sphere, so it differs from the model's own areas by well under a
       percent for an ocean grid.
    3. spherical areas from 1D latitude/longitude bounds, for a regular
       grid.

    Only if all three fail does this raise, and then with instructions.
    """
    horizontal = horizontal_slice(cube)

    for measure in horizontal.cell_measures():
        if measure.var_name in ("areacello", "areacella", "areacell"):
            return np.ma.masked_invalid(
                np.ma.asarray(
                    horizontal.cell_measure(measure.name()).core_data(),
                    dtype=float).reshape(horizontal.shape))

    from_bounds = area_from_bounds(horizontal)
    if from_bounds is not None:
        return from_bounds

    if horizontal.coord("latitude").ndim > 1:
        raise ValueError(
            "this cube has 2D (curvilinear) latitude/longitude and no cell "
            "corner bounds, so no cell area can be obtained. Either attach "
            "areacello -- in a recipe with\n"
            "    supplementary_variables:\n"
            "      - short_name: areacello\n"
            "        mip: Ofx\n"
            "(add 'exp: piControl' if the experiment you want does not "
            "publish it), in a notebook with "
            "dataset.add_supplementary(short_name='areacello', mip='Ofx') "
            "-- or keep the coordinate bounds, which CMOR requires and "
            "which are enough to compute the area exactly.")

    # iris defaults to an Earth radius of 6367470 m where the cube has
    # no coordinate system, while cell_lengths uses the mean radius of
    # 6371000 m. That is a 0.11% difference in area, small but free to
    # avoid: pin the same sphere so lengths and areas agree.
    horizontal = horizontal.copy()
    guess_bounds(horizontal)
    if horizontal.coord("latitude").coord_system is None:
        sphere = iris.coord_systems.GeogCS(EARTH_RADIUS)
        horizontal.coord("latitude").coord_system = sphere
        horizontal.coord("longitude").coord_system = sphere
    return iris.analysis.cartography.area_weights(horizontal)


def cell_lengths(cube):
    """Zonal and meridional cell lengths in metres.

    Returns ``(dx, dy)`` broadcast to the horizontal shape of ``cube``.
    ``dx`` shrinks with the cosine of latitude, ``dy`` does not.
    """
    lat = lat_1d(cube)
    lon = lon_1d(cube)
    dlon = np.gradient(lon) * np.pi / 180.0
    dlat = np.gradient(lat) * np.pi / 180.0
    coslat = np.cos(np.deg2rad(lat))
    dx = EARTH_RADIUS * coslat[:, None] * dlon[None, :]
    dy = EARTH_RADIUS * np.broadcast_to(dlat[:, None], dx.shape)
    return dx, np.array(dy)


def horizontal_gradient(field, cube, periodic=True):
    """Central-difference gradient of ``field`` in m-1 units.

    ``field`` is a 2D masked array on the horizontal grid of ``cube``.
    Returns ``(d/dx, d/dy)``.  The zonal direction wraps when
    ``periodic`` is set, which is correct for a global CMOR grid.
    """
    dxm, dym = cell_lengths(cube)
    data = np.ma.masked_invalid(np.ma.asarray(field, dtype=float))
    filled = data.filled(np.nan)

    if periodic:
        padded = np.concatenate(
            [filled[:, -1:], filled, filled[:, :1]], axis=1)
        dfdx = (padded[:, 2:] - padded[:, :-2]) / (2.0 * dxm)
    else:
        dfdx = np.full_like(filled, np.nan)
        dfdx[:, 1:-1] = (filled[:, 2:] - filled[:, :-2]) / (2.0 * dxm[:, 1:-1])

    dfdy = np.full_like(filled, np.nan)
    dfdy[1:-1, :] = (filled[2:, :] - filled[:-2, :]) / (2.0 * dym[1:-1, :])

    return np.ma.masked_invalid(dfdx), np.ma.masked_invalid(dfdy)


def topographic_unit_vectors(deptho, cube):
    """Unit vectors tangent and normal to the bathymetry contours.

    Returns ``(t_x, t_y, n_x, n_y)``.  The normal points towards deeper
    water; the tangent is the normal rotated so that shallow water lies
    to the left, matching the sketch in
    ``Along-slope-velocities.ipynb``:

        u_along = u * h_y/|grad h| - v * h_x/|grad h|
        v_cross = u * h_x/|grad h| + v * h_y/|grad h|
    """
    dhdx, dhdy = horizontal_gradient(deptho, cube)
    magnitude = np.ma.sqrt(dhdx**2 + dhdy**2)
    magnitude = np.ma.masked_where(magnitude == 0, magnitude)
    n_x = dhdx / magnitude
    n_y = dhdy / magnitude
    return n_y, -n_x, n_x, n_y


def project_along_slope(u_field, v_field, deptho, cube):
    """Along- and cross-slope velocity components."""
    t_x, t_y, n_x, n_y = topographic_unit_vectors(deptho, cube)
    along = u_field * t_x + v_field * t_y
    cross = u_field * n_x + v_field * n_y
    return along, cross


def bin_centres(bins):
    """Midpoints of a monotonic array of bin edges."""
    bins = np.asarray(bins, dtype=float)
    return 0.5 * (bins[1:] + bins[:-1])


def bin_sum(values, bin_variable, bins, axis=None):
    """Sum ``values`` into ``bins`` of ``bin_variable``.

    ``values`` and ``bin_variable`` must have the same shape.  Returns
    an array whose first axis runs over the bin centres and whose
    remaining axes are the axes of ``values`` *not* listed in ``axis``.
    With ``axis=None`` everything is collapsed into a 1D profile.

    This is ``xhistogram.xarray.histogram(bin_variable, bins=bins,
    weights=values, dim=axis)`` without the xarray dependency.
    """
    values = np.ma.masked_invalid(np.ma.asarray(values, dtype=float))
    bin_variable = np.ma.masked_invalid(
        np.ma.asarray(bin_variable, dtype=float))
    bins = np.asarray(bins, dtype=float)
    n_bins = bins.size - 1

    if axis is None:
        axis = tuple(range(values.ndim))
    elif isinstance(axis, int):
        axis = (axis,)
    axis = tuple(a % values.ndim for a in axis)
    keep = tuple(a for a in range(values.ndim) if a not in axis)

    moved = np.ma.transpose(values, keep + axis)
    moved_bin = np.ma.transpose(bin_variable, keep + axis)
    kept_shape = moved.shape[:len(keep)]
    flat = moved.reshape(int(np.prod(kept_shape)) if keep else 1, -1)
    flat_bin = moved_bin.reshape(flat.shape)

    out = np.zeros((n_bins,) + kept_shape)
    flat_out = out.reshape(n_bins, -1)
    # np.digitize returns 0 below the first edge and n_bins+1 above the
    # last, so a bin index of 1..n_bins is inside the range.
    for column in range(flat.shape[0]):
        data = flat[column]
        coordinate = flat_bin[column]
        valid = ~np.ma.getmaskarray(data) & ~np.ma.getmaskarray(coordinate)
        if not valid.any():
            continue
        index = np.digitize(
            np.ma.filled(coordinate, np.nan)[valid], bins)
        inside = (index >= 1) & (index <= n_bins)
        flat_out[:, column] = np.bincount(
            index[inside] - 1,
            weights=np.ma.filled(data, 0.0)[valid][inside],
            minlength=n_bins)
    return out

## Compute

In [ ]:
ISOBATH_BINS = np.array([0., 500., 1000., 1500., 2000., 3000.,
                         4000., 6000.])
MAX_LATITUDE = -59.0

u_surface = surface_field(uo_cube)
v_surface = surface_field(vo_cube)
depth = masked_data(deptho_cube)

along, cross = project_along_slope(
    masked_data(u_surface), masked_data(v_surface), depth,
    u_surface)

southern_mask = lat_2d(u_surface) < MAX_LATITUDE
along_south = np.ma.masked_where(~southern_mask, along)
depth_south = np.ma.masked_where(~southern_mask, depth)

# Area-weighted mean of the along-slope flow per isobath band.
area = cell_area(u_surface)
weights = np.ma.masked_where(np.ma.getmaskarray(along_south), area)
weighted = bin_sum(along_south * weights, depth_south,
                   ISOBATH_BINS)
totals = bin_sum(weights, depth_south, ISOBATH_BINS)
with np.errstate(divide='ignore', invalid='ignore'):
    profile = np.ma.masked_invalid(
        np.where(totals > 0, weighted / totals, np.nan))

## Plot

In [ ]:
fig = plt.figure(figsize=(15, 5.5))
map_axes = fig.add_subplot(1, 2, 1,
                           projection=ccrs.SouthPolarStereo())
profile_axes = fig.add_subplot(1, 2, 2)
scale = np.ma.max(np.ma.abs(along_south))

# The original recipe presents this as an Antarctic map.  A
# rectangle makes the circumpolar current look discontinuous at
# the longitude seam and was the source of the apparent mismatch.
mesh = map_axes.pcolormesh(lon_1d(u_surface), lat_1d(u_surface),
                           along_south, cmap='RdBu_r',
                           shading='auto', vmin=-scale, vmax=scale,
                           transform=ccrs.PlateCarree())
map_axes.set_extent([-180., 180., -82., -55.],
                    crs=ccrs.PlateCarree())
map_axes.gridlines(linewidth=0.4, alpha=0.4)
map_axes.set_title('Along-slope surface velocity')
fig.colorbar(mesh, ax=map_axes, shrink=0.8, label='m s$^{-1}$')

profile_axes.plot(profile, bin_centres(ISOBATH_BINS), marker='o')
profile_axes.axvline(0, color='k', linewidth=0.6)
profile_axes.invert_yaxis()
profile_axes.set_xlabel('Along-slope velocity (m s$^{-1}$)')
profile_axes.set_ylabel('Bathymetry (m)')
profile_axes.set_title(f'By isobath, south of {MAX_LATITUDE:.0f}'
                       r'$^\circ$')
profile_axes.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()